In [0]:
from pyspark.sql.functions import col, desc, explode, split, avg, count, regexp_replace

# Top 15 box-office all-time
# ============================================
(spark.table("workspace.default.films_silver")
    .filter(col("revenue").isNotNull())
    .orderBy(desc("revenue")).limit(15)
    .write.format("delta").mode("overwrite")
    .saveAsTable("workspace.default.gold_top_boxoffice"))


#  Budget vs Revenue (échantillon pour le scatter)
# ============================================
(spark.table("workspace.default.films_silver")
    .filter(col("budget").isNotNull() & col("revenue").isNotNull())
    .sample(fraction=0.15, seed=42)
    .write.format("delta").mode("overwrite")
    .saveAsTable("workspace.default.gold_budget_vs_revenue"))

# Genres par décennie (top 5 gérés côté viz)
# ============================================
(spark.table("workspace.default.films_silver")
    .filter(col("genres_list").isNotNull() & col("decennie").isNotNull())
    .withColumn("genre", explode(split(regexp_replace("genres_list", "[\\[\\]']", ""), ",\\s*")))
    .groupBy("decennie", "genre").count()
    .write.format("delta").mode("overwrite")
    .saveAsTable("workspace.default.gold_genres_by_decade"))


# Stats annuelles (note moyenne + volume)
# ============================================
(spark.table("workspace.default.films_silver")
    .filter(col("annee").isNotNull() & (col("annee") >= 1980))
    .groupBy("annee")
    .agg(avg("vote_average").alias("note_moy"),
         count("*").alias("nb_films"))
    .write.format("delta").mode("overwrite")
    .saveAsTable("workspace.default.gold_yearly_stats"))

# Top 10 films par note (min. 5k votes)
# ============================================
(spark.table("workspace.default.films_silver")
    .filter((col("vote_count") > 5000) & col("vote_average").isNotNull())
    .orderBy(desc("vote_average")).limit(10)
    .write.format("delta").mode("overwrite")
    .saveAsTable("workspace.default.gold_top_rated"))

#Matrice de corrélation (échantillon)
# ============================================
(spark.table("workspace.default.films_silver")
    .select("budget","revenue","popularity","runtime","vote_average","vote_count")
    .dropna().sample(fraction=0.2, seed=42)
    .write.format("delta").mode("overwrite")
    .saveAsTable("workspace.default.gold_correlation"))

print(" Gold tables créées")